# Джем MyIndie LVL 10

# 0. Подготовка

In [4]:
import os
from datetime import datetime
# from pathlib import Path
# import shutil
import json
import requests
import re
import pandas as pd
import numpy as np


# Константы
MYINDIE_JAM_URL = "https://myindie.ru/jams/jam/myindie-level-10"
MYINDIE_JAM_URL_PAGES = 5  # Количество страниц с играми на геймджеме
MYINDIE_JAM_SKIP_DOWNLOAD = False
OUTPUT_DIR = "output/"
CRITERIAS = ["art", "sound", "theme", "devblog", "gameplay", "narrative", "decoration", "overall_impression"]  # Список критериев для анализа

# Директория для html-файлов джема
jam_path = MYINDIE_JAM_URL.split('/')[-1]  # получаем имя джема вроде "myindie-game-jam-level-9"
if jam_path is None or jam_path == "":
    raise ValueError("Не удалось извлечь путь джема из URL. Проверьте правильность MYINDIE_JAM_URL.")
os.makedirs(os.path.join(OUTPUT_DIR, jam_path), exist_ok=True)

pd.set_option('display.float_format', '{:.3f}'.format)


def get_jam_games_urls(jam_url):
    """ Скачивает страницу списка игр джема и извлекает URL игр. """

    response = requests.get(jam_url)

    if response.status_code != 200:
        print(f"Ошибка при получении страницы джема: `{response.status_code}`")
        return []

    games_urls = re.findall(r'/games/game/[\w-]+', response.text)
    games_urls = [f"https://myindie.ru{url}" for url in games_urls]
    print(f"Найдено {len(games_urls)} URL игр в: {jam_url}")

    return games_urls


def unflatten_nuxt_data(data):
    """ Разворачивает плоскую структуру данных Nuxt.js в дерево. """
    if not isinstance(data, list) or not data: return data
    memo = {}
    def resolve(val):
        if isinstance(val, int) and 0 <= val < len(data):
            if val not in memo:
                memo[val] = resolve_item(data[val])
            return memo[val]
        return val
    def resolve_item(item):
        if isinstance(item, dict):
            return {k: resolve(v) for k, v in item.items()}
        if isinstance(item, list):
            return [resolve(v) for v in item]
        return item
    return resolve_item(data[1])

# 1. Скачиваем все страницы игр геймджема

1.1 Ищем все игры джема

In [5]:
if not MYINDIE_JAM_SKIP_DOWNLOAD:
    games_urls = []
    for page in range(1, MYINDIE_JAM_URL_PAGES + 1):
        paged_url = f"{MYINDIE_JAM_URL}/games?page={page}"
        print(f"Обрабатываем страницу: {paged_url}")
        urls = get_jam_games_urls(paged_url)
        games_urls.extend(urls)

    print(f"\nНайдено {len(games_urls)} игр на {MYINDIE_JAM_URL_PAGES} страницах:\n{chr(10).join(games_urls)}")

Обрабатываем страницу: https://myindie.ru/jams/jam/myindie-level-10/games?page=1
Найдено 30 URL игр в: https://myindie.ru/jams/jam/myindie-level-10/games?page=1
Обрабатываем страницу: https://myindie.ru/jams/jam/myindie-level-10/games?page=2
Найдено 30 URL игр в: https://myindie.ru/jams/jam/myindie-level-10/games?page=2
Обрабатываем страницу: https://myindie.ru/jams/jam/myindie-level-10/games?page=3
Найдено 31 URL игр в: https://myindie.ru/jams/jam/myindie-level-10/games?page=3
Обрабатываем страницу: https://myindie.ru/jams/jam/myindie-level-10/games?page=4
Найдено 30 URL игр в: https://myindie.ru/jams/jam/myindie-level-10/games?page=4
Обрабатываем страницу: https://myindie.ru/jams/jam/myindie-level-10/games?page=5
Найдено 23 URL игр в: https://myindie.ru/jams/jam/myindie-level-10/games?page=5

Найдено 144 игр на 5 страницах:
https://myindie.ru/games/game/my-tedious-routine
https://myindie.ru/games/game/manlygun
https://myindie.ru/games/game/koronka-zhmyot
https://myindie.ru/games/game

1.2 Скачать все HTML-страницы игр геймджема

In [ ]:
if not MYINDIE_JAM_SKIP_DOWNLOAD:
    jam_games_file_path = os.path.join(OUTPUT_DIR, jam_path, f"jam_games.txt")
    if os.path.exists(jam_games_file_path):
        os.remove(jam_games_file_path)
    jam_games_file = open(jam_games_file_path, 'a', encoding='utf-8')

    # DEBUG_GAMES_MAX = 3  # delete
    for i, game_url in enumerate(games_urls): #[:DEBUG_GAMES_MAX]):
        print(f"Обработка URL игры: {game_url}")
        response = requests.get(game_url)
        if response.status_code != 200:
            print(f"* Ошибка при получении страницы игры: `{response.status_code}`")
            continue

        title_match = re.search(r'<title>([^<]+)</title>', response.text)
        if title_match:
            title = f"{i:03d}_" + re.sub(r'[^\w_]', '-', re.sub(r'\s+', '_', title_match.group(1).strip()))
        else:
            title = f"{i:03d}_" + "Unknown"

        output_file_path = os.path.join(OUTPUT_DIR, jam_path, f"{title}.html")
        with open(output_file_path, 'w', encoding='utf-8') as f:
            f.write(response.text)
            print(f"Сохранено в `{output_file_path}`")
        jam_games_file.write(f"{output_file_path} {game_url}\n")

    jam_games_file.close()

Обработка URL игры: https://myindie.ru/games/game/my-tedious-routine
Сохранено в `output/myindie-level-10\000_My_Tedious_Routine-_Жанр-_Arcade-_Action_-_Инди-игры_-_MyIndie.html`
Обработка URL игры: https://myindie.ru/games/game/manlygun
Сохранено в `output/myindie-level-10\001_ManlyGun-_Жанр-_Shooter_-_Инди-игры_-_MyIndie.html`
Обработка URL игры: https://myindie.ru/games/game/koronka-zhmyot
Сохранено в `output/myindie-level-10\002_Коронка_Жмёт--_Жанр-_Arcade_-_Инди-игры_-_MyIndie.html`


# 2. Извлекаем данные со всех страниц игр
2.1 Используем уже скачанные HTML-страницы игр геймджема, чтобы извлечь данные о каждой игре

In [26]:
jam_games_file_path = os.path.join(OUTPUT_DIR, jam_path, "jam_games.txt")
jam_games_file = open(jam_games_file_path, 'r', encoding='utf-8')
print(f"Всего игр в файле `{jam_games_file_path}`: {len(jam_games_file.readlines())}")

Всего игр в файле `output/myindie-level-10\jam_games.txt`: 3


2.2 Собираем оценки и отзывы для каждой игры из скачанных файлов

In [27]:
all_judges_reviews = []  # без разделения по играм
all_participant_reviews = []  # без разделения по играм
all_users_reviews = []  # без разделения по играм
game_reviews = {}  # отзывы для каждой игры, ключ - alias игры, значение - словарь с отзывами по типам

jam_games_file.seek(0)

for line in jam_games_file:
    output_file_path, game_url = line.strip().split(' ', 1)
    alias = game_url.split('/')[-1]
    print(f"Обработка: {game_url}")
    with open(output_file_path, 'r', encoding='utf-8') as f:
        html_content = f.read()

    match = re.search(r'id=\"__NUXT_DATA__\">([^<]+)</script>', html_content)
    if not match:
        print(f"* Не найден __NUXT_DATA__ в `{output_file_path}`\n")
        continue

    json_data = json.loads(match.group(1))
    unflattened = unflatten_nuxt_data(json_data)
    if not unflattened:
        print(f"* Не удалось развернуть данные Nuxt.js в `{output_file_path}`\n")
        continue

    data_section = unflattened.get('data', [])
    if not data_section:
        print(f"* Не найден раздел 'data' в развернутых данных Nuxt.js в `{output_file_path}`\n")
        continue

    # Берем второй элемент списка (индекс 1) — там словарь с результатами
    if isinstance(data_section, list) and len(data_section) > 1:
        payload_container = data_section[1]
        if not payload_container:
            print(f"* Пустой контейнер в `{output_file_path}`\n")
            continue

        # В словаре берем первый ключ (game<alias>)
        if isinstance(payload_container, dict) and payload_container:
            second_key = list(payload_container.keys())[0]
            games = payload_container[second_key]
            if not games:
                print(f"* Пустой объект игры в `{output_file_path}`\n")
                continue

            # Извлекаем отзывы из game_payload['data']['reviews']
            if isinstance(games, dict):
                inner_data = games.get('data', {})
                if not inner_data:
                    print(f"* Пустой объект 'data' в `{output_file_path}`\n")
                    continue

                reviews = inner_data.get('reviews', [])
                if not reviews:
                    print(f"* Не найдено отзывов в `{output_file_path}`\n")
                    continue
                # print(json.dumps(reviews, ensure_ascii=False, indent=2))

                judges = [r for r in reviews if isinstance(r, dict) and r.get('reviewerType') == 'judge' and r.get('user') is not None]
                all_judges_reviews.extend(judges)
                participants = [r for r in reviews if isinstance(r, dict) and r.get('reviewerType') == 'jam-participant' and r.get('user') is not None]
                all_participant_reviews.extend(participants)
                users = [r for r in reviews if isinstance(r, dict) and r.get('reviewerType') == 'user' and r.get('user') is not None]
                all_users_reviews.extend(users)

                game_reviews[alias] = {
                    'judges': judges,
                    'participants': participants,
                    'users': users,
                }
                print(f"Судейских отзывов: {len(judges)} - {', '.join([r.get('user', {}).get('username') for r in judges])}")
                print(f"Отзывы участников: {len(participants)} - {', '.join([r.get('user', {}).get('username') for r in participants])}")
                print(f"Отзывы пользователей: {len(users)} - {', '.join([r.get('user', {}).get('username') for r in users])}\n")

print(f"\nВсего судейских отзывов на джеме: {len(all_judges_reviews)}")
print(f"Всего отзывов участников на джеме: {len(all_participant_reviews)}")
print(f"Всего отзывов пользователей на джеме: {len(all_users_reviews)}")

Обработка: https://myindie.ru/games/game/my-tedious-routine
Судейских отзывов: 3 - Banzai, Vlad_Borets, StrasZakata
Отзывы участников: 4 - shishinda, SkreT1kk, SPIRONIS, Gazolina
Отзывы пользователей: 0 - 

Обработка: https://myindie.ru/games/game/manlygun
Судейских отзывов: 4 - DenisTrak, Alex_app1, Markus_Glevera, scrabyq
Отзывы участников: 2 - Sdt, shishinda
Отзывы пользователей: 0 - 

Обработка: https://myindie.ru/games/game/koronka-zhmyot
Судейских отзывов: 3 - DenisTrak, Vlad_Borets, Markus_Glevera
Отзывы участников: 2 - UnrealQW, shishinda
Отзывы пользователей: 0 - 


Всего судейских отзывов на джеме: 10
Всего отзывов участников на джеме: 8
Всего отзывов пользователей на джеме: 0


In [40]:
# game_reviews

In [28]:
jam_games_file.close()

# 3. Обработка данных и анализ

In [54]:
# print(json.dumps(all_judges_reviews, ensure_ascii=False, indent=2))
# print(json.dumps(all_participant_reviews, ensure_ascii=False, indent=2))
# print(json.dumps(all_users_reviews, ensure_ascii=False, indent=2))

3.1 Обрабатываем все судейские отзывы и оценки для каждой игры

In [29]:
# Словари для хранения данных судей
judges_dict = {} # userId -> {"username": username, "reviews": []}

for review in all_judges_reviews:
    if review.get('reviewerType') == 'judge':
        user_id = review.get('userId')
        username = review.get('user', {}).get('username', 'Unknown')

        if user_id not in judges_dict:
            judges_dict[user_id] = {
                "username": username,
                "reviews": []
            }

        judges_dict[user_id]["reviews"].append(review)

judges_dict = dict(sorted(judges_dict.items(), key=lambda item: len(item[1]['reviews']), reverse=True))

print(f"Количество уникальных судей: {len(judges_dict)}")
for user_id, info in judges_dict.items():
    print(f"Судья: {info['username']} (ID: {user_id}) - Обзоров: {len(info['reviews'])}")

# Финальный словарь с идентификацией по username
judges_reviews_by_username = {info['username']: info['reviews'] for info in judges_dict.values()}

Количество уникальных судей: 7
Судья: Vlad_Borets (ID: 5797a548-ec39-4b96-a34c-684b7b41071d) - Обзоров: 2
Судья: DenisTrak (ID: 1b4d8917-8493-4dcd-96c8-1e7748137dab) - Обзоров: 2
Судья: Markus_Glevera (ID: 891b5229-ce62-4f23-a950-bd7579ef4731) - Обзоров: 2
Судья: Banzai (ID: 195c769e-c340-4bd7-bda1-a688e332ec9d) - Обзоров: 1
Судья: StrasZakata (ID: 431cb7d7-9108-468b-890e-6befedb80325) - Обзоров: 1
Судья: Alex_app1 (ID: 91d843a5-c489-4aea-ac13-e26907a673ab) - Обзоров: 1
Судья: scrabyq (ID: 54c7df1b-c80e-4dc6-a11d-0fd2c328f349) - Обзоров: 1


In [44]:
# print(json.dumps(judges_reviews_by_username, ensure_ascii=False, indent=2))

3.1.1 Обрабатываем все отзывы участников джема и оценки для каждой игры

In [30]:
# Словари для хранения данных участников джема
participants_dict = {} # userId -> {"username": username, "reviews": []}

for review in all_participant_reviews:
    if review.get('reviewerType') == 'jam-participant':
        user_id = review.get('userId')
        username = review.get('user', {}).get('username', 'Unknown')

        if user_id not in participants_dict:
            participants_dict[user_id] = {
                "username": username,
                "reviews": []
            }

        participants_dict[user_id]["reviews"].append(review)

participants_dict = dict(sorted(participants_dict.items(), key=lambda item: len(item[1]['reviews']), reverse=True))

print(f"Количество уникальных участников джема: {len(participants_dict)}")
for user_id, info in participants_dict.items():
    print(f"Участник джема: {info['username']} (ID: {user_id}) - Обзоров: {len(info['reviews'])}")

# Финальный словарь с идентификацией по username
participants_reviews_by_username = {info['username']: info['reviews'] for info in participants_dict.values()}

Количество уникальных участников джема: 6
Участник джема: shishinda (ID: a4e48532-b463-4585-addf-3865767100f0) - Обзоров: 3
Участник джема: SkreT1kk (ID: 7606e2ea-1b4d-435f-bce3-dfacd55ddcb4) - Обзоров: 1
Участник джема: SPIRONIS (ID: 98cf62c2-6635-4ef9-9e5f-d66e3dc91db6) - Обзоров: 1
Участник джема: Gazolina (ID: 0d7c6bc5-49b1-47f2-9947-3742d69c2d0d) - Обзоров: 1
Участник джема: Sdt (ID: c9a5a47c-629b-4988-8b4f-8214a9eb4a09) - Обзоров: 1
Участник джема: UnrealQW (ID: cceb79e8-01c8-4133-8733-4fdfbe290cf2) - Обзоров: 1


In [46]:
# print(json.dumps(participants_reviews_by_username, ensure_ascii=False, indent=2))

3.2 Собираем статистику по каждому судье: средний балл, количество оценок, количество отзывов

In [31]:
judges_stats = []

for username, reviews in judges_reviews_by_username.items():
    if not reviews:
        print(f"* Судья {username} не имеет обзоров, пропускаем.")
        continue

    totals = { 'score': [] }
    for criteria in CRITERIAS:
        totals[criteria] = []

    for r in reviews:
        if 'score' in r:
            totals['score'].append(r['score'])

        c = r.get('criterias', {})
        for criteria in CRITERIAS:
            if criteria in c:
                totals[criteria].append(c[criteria])

    judge_row = {
        'username': username,
        'reviews_count': len(reviews)
    }

    for key, values in totals.items():
        judge_row[f'avg_{key}'] = round(sum(values) / len(values), 3) if values else 0

    judges_stats.append(judge_row)

df_judges_stats = pd.DataFrame(judges_stats)

# Вывод результата, отсортированного по среднему баллу
df_judges_stats.sort_values(by='avg_score', ascending=False)


,username,reviews_count,avg_score,avg_art,avg_sound,avg_theme,avg_devblog,avg_gameplay,avg_narrative,avg_decoration,avg_overall_impression
4,StrasZakata,1,3.100,4.000,4.000,5.000,1.000,3.000,3.000,1.000,4.000
1,DenisTrak,2,2.750,2.500,2.750,2.750,3.000,2.500,3.000,3.000,2.500
0,Vlad_Borets,2,2.400,3.250,2.250,2.750,1.750,2.000,3.000,2.000,2.000
2,Markus_Glevera,2,2.200,2.750,2.250,1.500,1.500,2.000,2.000,2.500,3.250
5,Alex_app1,1,2.100,3.000,2.000,1.000,2.500,3.000,1.000,1.500,2.500
3,Banzai,1,1.900,2.000,1.500,4.000,1.000,2.000,3.000,1.000,1.000
6,scrabyq,1,1.600,2.000,2.000,1.000,2.000,1.500,1.000,1.500,1.500


3.2.1 Собираем статистику по каждому участнику джема: средний балл, количество оценок, количество отзывов

In [32]:
participants_stats = []

for username, reviews in participants_reviews_by_username.items():
    if not reviews:
        print(f"* Участник джема {username} не имеет обзоров, пропускаем.")
        continue

    totals = { 'score': [] }
    for criteria in CRITERIAS:
        totals[criteria] = []

    for r in reviews:
        if 'score' in r:
            totals['score'].append(r['score'])

        c = r.get('criterias', {})
        for criteria in CRITERIAS:
            if criteria in c:
                totals[criteria].append(c[criteria])

    participant_row = {
        'username': username,
        'reviews_count': len(reviews)
    }

    for key, values in totals.items():
        participant_row[f'avg_{key}'] = round(sum(values) / len(values), 3) if values else 0

    participants_stats.append(participant_row)

df_participants_stats = pd.DataFrame(participants_stats)

# Вывод результата, отсортированного по среднему баллу
df_participants_stats.sort_values(by='avg_score', ascending=False)

,username,reviews_count,avg_score,avg_art,avg_sound,avg_theme,avg_devblog,avg_gameplay,avg_narrative,avg_decoration,avg_overall_impression
5,UnrealQW,1,3.900,4.000,3.000,3.000,5.000,3.000,5.000,5.000,3.000
3,Gazolina,1,2.900,4.000,3.500,4.000,1.500,2.000,4.000,1.500,2.500
2,SPIRONIS,1,2.500,4.000,3.500,2.500,1.000,1.000,4.000,1.000,3.000
1,SkreT1kk,1,2.300,3.000,3.000,2.000,1.000,3.500,2.500,1.000,2.500
4,Sdt,1,2.100,3.000,3.000,1.000,4.000,3.000,1.000,1.000,1.000
0,shishinda,3,1.867,2.167,2.333,1.333,2.000,1.500,1.667,2.000,1.833


3.3 Cтатистический анализ: средние оценки, медианы, стандартные отклонения, распределения оценок и длины отзывов

In [33]:
judge_detailed_stats = []

for username, reviews in judges_reviews_by_username.items():
    judge_precise_scores = []
    judge_review_lengths = []

    for r in reviews:
        c = r.get('criterias', {})
        crit_values = [c[k] for k in CRITERIAS if k in c]

        if crit_values:
            game_score = sum(crit_values) / len(crit_values)
            judge_precise_scores.append(game_score)

        text = r.get('reviewText', '')
        if text:
            clean_text = re.sub(r'<[^>]+>', '', text)
            judge_review_lengths.append(len(clean_text))

    stats = {
        'username': username,
        'min_score': np.min(judge_precise_scores) if judge_precise_scores else 0.0,
        'max_score': np.max(judge_precise_scores) if judge_precise_scores else 0.0,
        'median_score': np.median(judge_precise_scores) if judge_precise_scores else 0.0,
        'std_score': np.std(judge_precise_scores) if judge_precise_scores else 0.0,
        'avg_review_chars': int(np.mean(judge_review_lengths)) if judge_review_lengths else 0.0,
        'min_review_chars': np.min(judge_review_lengths) if judge_review_lengths else 0,
        'max_review_chars': np.max(judge_review_lengths) if judge_review_lengths else 0
    }
    judge_detailed_stats.append(stats)

df_judge_detailed = pd.DataFrame(judge_detailed_stats)

df_judge_final_stats = pd.merge(df_judges_stats, df_judge_detailed, on='username')

cols = ['username', 'reviews_count', 'avg_score', 'median_score', 'std_score', 'min_score', 'max_score', 'avg_review_chars', 'min_review_chars', 'max_review_chars']
df_judge_final_stats[cols].sort_values(by='avg_score', ascending=False)

,username,reviews_count,avg_score,median_score,std_score,min_score,max_score,avg_review_chars,min_review_chars,max_review_chars
4,StrasZakata,1,3.100,3.125,0.000,3.125,3.125,439,439,439
1,DenisTrak,2,2.750,2.750,0.625,2.125,3.375,135,27,244
0,Vlad_Borets,2,2.400,2.375,0.188,2.188,2.562,632,614,651
2,Markus_Glevera,2,2.200,2.219,0.219,2.000,2.438,3068,2402,3734
5,Alex_app1,1,2.100,2.062,0.000,2.062,2.062,118,118,118
3,Banzai,1,1.900,1.938,0.000,1.938,1.938,63,63,63
6,scrabyq,1,1.600,1.562,0.000,1.562,1.562,218,218,218


3.3.1 Cтатистический анализ: средние оценки, медианы, стандартные отклонения, распределения оценок и длины отзывов

In [34]:
participants_detailed_stats = []

for username, reviews in participants_reviews_by_username.items():
    participant_precise_scores = []
    participant_review_lengths = []

    for r in reviews:
        c = r.get('criterias', {})
        crit_values = [c[k] for k in CRITERIAS if k in c]

        if crit_values:
            game_score = sum(crit_values) / len(crit_values)
            participant_precise_scores.append(game_score)

        text = r.get('reviewText', '')
        if text:
            clean_text = re.sub(r'<[^>]+>', '', text)
            participant_review_lengths.append(len(clean_text))

    stats = {
        'username': username,
        'min_score': np.min(participant_precise_scores) if participant_precise_scores else 0.0,
        'max_score': np.max(participant_precise_scores) if participant_precise_scores else 0.0,
        'median_score': np.median(participant_precise_scores) if participant_precise_scores else 0.0,
        'std_score': np.std(participant_precise_scores) if participant_precise_scores else 0.0,
        'avg_review_chars': int(np.mean(participant_review_lengths)) if participant_review_lengths else 0.0,
        'min_review_chars': np.min(participant_review_lengths) if participant_review_lengths else 0,
        'max_review_chars': np.max(participant_review_lengths) if participant_review_lengths else 0
    }
    participants_detailed_stats.append(stats)

df_participants_detailed = pd.DataFrame(participants_detailed_stats)

df_participants_final_stats = pd.merge(df_participants_stats, df_participants_detailed, on='username')

cols = ['username', 'reviews_count', 'avg_score', 'median_score', 'std_score', 'min_score', 'max_score', 'avg_review_chars', 'min_review_chars', 'max_review_chars']
df_participants_final_stats[cols].sort_values(by='avg_score', ascending=False)

,username,reviews_count,avg_score,median_score,std_score,min_score,max_score,avg_review_chars,min_review_chars,max_review_chars
5,UnrealQW,1,3.900,3.875,0.000,3.875,3.875,731.000,731,731
3,Gazolina,1,2.900,2.875,0.000,2.875,2.875,243.000,243,243
2,SPIRONIS,1,2.500,2.500,0.000,2.500,2.500,317.000,317,317
1,SkreT1kk,1,2.300,2.312,0.000,2.312,2.312,0.000,0,0
4,Sdt,1,2.100,2.125,0.000,2.125,2.125,63.000,63,63
0,shishinda,3,1.867,1.562,0.744,1.125,2.875,614.000,31,1065


3.4 Больше статистики судей:
* strictness_index (индекс "строгости"): Отрицательный = строгий, Положительный = добрый

Строгость считается от среднего значения оценок, выставленных судьями. Если судья ставит в среднем низкие оценки, то он считается строгим, если высокие -- добрым. Этот индекс может быть полезен для анализа "везучести" игр.

In [35]:
judges_deep_insights = []

all_judges_precise_scores = []
for reviews in judges_reviews_by_username.values():
    for r in reviews:
        c = r.get('criterias', {})
        crit_values = [c[k] for k in CRITERIAS if k in c]
        if crit_values:
            all_judges_precise_scores.append(sum(crit_values) / len(crit_values))

judges_global_avg = np.mean(all_judges_precise_scores) if all_judges_precise_scores else 0.0

for username, reviews in judges_reviews_by_username.items():
    if not reviews: continue

    judge_precise_scores = []
    judge_review_dates = []

    # Статистика по критериям для этого судьи
    criteria_totals = {k: [] for k in CRITERIAS}

    for r in reviews:
        c = r.get('criterias', {})
        cv = [c[k] for k in CRITERIAS if k in c]
        if cv:
            judge_precise_scores.append(sum(cv)/len(cv))

        for k in CRITERIAS:
            if k in c: criteria_totals[k].append(c[k])

        # Время
        dt_str = r.get('createdAt')
        if dt_str:
            dt = datetime.strptime(dt_str, "%Y-%m-%dT%H:%M:%S.%fZ")
            judge_review_dates.append(dt.date())

    # любимый и нелюбимый критерий (avg)
    crit_avgs = {k: np.mean(v) if v else 0 for k, v in criteria_totals.items()}
    best_crit = max(crit_avgs, key=crit_avgs.get)
    worst_crit = min(crit_avgs, key=crit_avgs.get)

    judge_avg = np.mean(judge_precise_scores) if judge_precise_scores else 0.0

    insight = {
        'username': username,
        'strictness_index': round(judge_avg - judges_global_avg, 3), # Отрицательный = строгий, Положительный = добрый
        'top_criteria': f"{best_crit} ({round(crit_avgs[best_crit], 2)})",
        'bottom_criteria': f"{worst_crit} ({round(crit_avgs[worst_crit], 2)})",
        'active_days': (max(judge_review_dates) - min(judge_review_dates)).days + 1 if judge_review_dates else 0,
        'reviews_per_day': round(len(reviews) / ((max(judge_review_dates) - min(judge_review_dates)).days + 1), 2) if judge_review_dates else 0
    }
    judges_deep_insights.append(insight)

df_judges_insights = pd.DataFrame(judges_deep_insights)

df_judges_insights.sort_values(by='strictness_index')


,username,strictness_index,top_criteria,bottom_criteria,active_days,reviews_per_day
6,scrabyq,-0.775,art (2.0),theme (1.0),1,1.000
3,Banzai,-0.400,theme (4.0),devblog (1.0),1,1.000
5,Alex_app1,-0.275,art (3.0),theme (1.0),1,1.000
2,Markus_Glevera,-0.119,overall_impression (3.25),theme (1.5),1,2.000
0,Vlad_Borets,0.038,art (3.25),devblog (1.75),1,2.000
1,DenisTrak,0.413,devblog (3.0),art (2.5),2,1.000
4,StrasZakata,0.788,theme (5.0),devblog (1.0),1,1.000


3.4.1 Больше статистики участников джема:
- strictness_index: Отрицательный = строгий, Положительный = добрый

In [36]:
participants_deep_insights = []

all_participants_precise_scores = []
for reviews in participants_reviews_by_username.values():
    for r in reviews:
        c = r.get('criterias', {})
        crit_values = [c[k] for k in CRITERIAS if k in c]
        if crit_values:
            all_participants_precise_scores.append(sum(crit_values) / len(crit_values))

participants_global_avg = np.mean(all_participants_precise_scores) if all_participants_precise_scores else 0.0

for username, reviews in participants_reviews_by_username.items():
    if not reviews: continue

    participant_precise_scores = []
    participant_review_dates = []

    # Статистика по критериям для этого участника
    criteria_totals = {k: [] for k in CRITERIAS}

    for r in reviews:
        c = r.get('criterias', {})
        cv = [c[k] for k in CRITERIAS if k in c]
        if cv:
            participant_precise_scores.append(sum(cv)/len(cv))

        for k in CRITERIAS:
            if k in c: criteria_totals[k].append(c[k])

        # Время
        dt_str = r.get('createdAt')
        if dt_str:
            dt = datetime.strptime(dt_str, "%Y-%m-%dT%H:%M:%S.%fZ")
            participant_review_dates.append(dt.date())

    # любимый и нелюбимый критерий (avg)
    crit_avgs = {k: np.mean(v) if v else 0 for k, v in criteria_totals.items()}
    best_crit = max(crit_avgs, key=crit_avgs.get)
    worst_crit = min(crit_avgs, key=crit_avgs.get)

    participant_avg = np.mean(participant_precise_scores) if participant_precise_scores else 0.0

    insight = {
        'username': username,
        'strictness_index': round(participant_avg - participants_global_avg, 3), # Отрицательный = строгий, Положительный = добрый
        'top_criteria': f"{best_crit} ({round(crit_avgs[best_crit], 2)})",
        'bottom_criteria': f"{worst_crit} ({round(crit_avgs[worst_crit], 2)})",
        'active_days': (max(participant_review_dates) - min(participant_review_dates)).days + 1 if participant_review_dates else 0,
        'reviews_per_day': round(len(reviews) / ((max(participant_review_dates) - min(participant_review_dates)).days + 1), 2) if participant_review_dates else 0
    }
    participants_deep_insights.append(insight)

df_participants_insights = pd.DataFrame(participants_deep_insights)

df_participants_insights.sort_values(by='strictness_index')


,username,strictness_index,top_criteria,bottom_criteria,active_days,reviews_per_day
0,shishinda,-0.552,sound (2.33),theme (1.33),3,1.000
4,Sdt,-0.281,devblog (4.0),theme (1.0),1,1.000
1,SkreT1kk,-0.094,gameplay (3.5),devblog (1.0),1,1.000
2,SPIRONIS,0.094,art (4.0),devblog (1.0),1,1.000
3,Gazolina,0.469,art (4.0),devblog (1.5),1,1.000
5,UnrealQW,1.469,devblog (5.0),sound (3.0),1,1.000


3.5 "Везучие" и "невезучие" игры

In [37]:
# Словари для быстрого поиска строгости
judge_strictness = df_judges_insights.set_index('username')['strictness_index'].to_dict()
participant_strictness = df_participants_insights.set_index('username')['strictness_index'].to_dict()

for alias, reviews in game_reviews.items():
    judges = reviews.get('judges', [])
    participants = reviews.get('participants', [])

    # Сбор данных по судьям
    j_info = []
    j_total_strictness = 0
    for r in judges:
        name = r.get('user', {}).get('username')
        strictness = judge_strictness.get(name, 0)
        j_info.append(f" {name} ({strictness})")
        j_total_strictness += strictness

    # Сбор данных по участникам
    p_info = []
    p_total_strictness = 0
    for r in participants:
        name = r.get('user', {}).get('username')
        strictness = participant_strictness.get(name, 0)
        p_info.append(f" {name} ({strictness})")
        p_total_strictness += strictness

    print(f"Игра: {alias}, URL: https://myindie.ru/games/game/{alias}")
    print(f"Судьи (строгость): {', '.join(j_info)}")
    print(f"Участники (строгость): {', '.join(p_info)}")
    print(f"Суммарная строгость судей: {round(j_total_strictness, 3)}")
    print(f"Суммарная строгость участников: {round(p_total_strictness, 3)}\n")

Игра: my-tedious-routine, URL: https://myindie.ru/games/game/my-tedious-routine
Судьи (строгость):  Banzai (-0.4),  Vlad_Borets (0.038),  StrasZakata (0.788)
Участники (строгость):  shishinda (-0.552),  SkreT1kk (-0.094),  SPIRONIS (0.094),  Gazolina (0.469)
Суммарная строгость судей: 0.426
Суммарная строгость участников: -0.083

Игра: manlygun, URL: https://myindie.ru/games/game/manlygun
Судьи (строгость):  DenisTrak (0.413),  Alex_app1 (-0.275),  Markus_Glevera (-0.119),  scrabyq (-0.775)
Участники (строгость):  Sdt (-0.281),  shishinda (-0.552)
Суммарная строгость судей: -0.756
Суммарная строгость участников: -0.833

Игра: koronka-zhmyot, URL: https://myindie.ru/games/game/koronka-zhmyot
Судьи (строгость):  DenisTrak (0.413),  Vlad_Borets (0.038),  Markus_Glevera (-0.119)
Участники (строгость):  UnrealQW (1.469),  shishinda (-0.552)
Суммарная строгость судей: 0.332
Суммарная строгость участников: 0.917

